In [1]:
import pandas as pd
import pyreadstat

def brand_weighted_mean(df, weight_var):
    weighted_df = df.select_dtypes(include = ['number', 'bool']).multiply(df[weight_var], axis=0)
    return weighted_df.sum() / df[weight_var].sum()

In [2]:
# Ingestion de datos
data, meta = pyreadstat.read_sav('36415052_EVENTOS_CERVEZAS_2025_Stacked_SPSS.sav')

In [3]:
# SOLO CORONA EVENTOS - FIX de escala TBCA Eventos 
columns_fix = data.filter(like='TBCA_Eventos_').columns
data[columns_fix] = data[columns_fix].map(lambda x: 0 if x == 2 else x)

# FIX Patrocinadores 
columns_fix_2 = data.filter(like='PATROCINADORES_SPSS').columns
data[columns_fix_2] = data[columns_fix_2].apply(lambda x: (x == data['Brand_ID']).astype(int))

# FIX SP1 Loop
fix_3 = pd.get_dummies(data['SP1Loop_SPSS'], prefix='SP1',prefix_sep="_", dummy_na=False).astype(int)
columns_fix_3 = fix_3.columns
data[columns_fix_3] = fix_3

C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26260\2570646652.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[columns_fix_3] = fix_3
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26260\2570646652.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[columns_fix_3] = fix_3
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26260\2570646652.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all colum

In [4]:
# Aplicar labels a variables categoricas
from h11 import Data
from traitlets import default


data['Respondent_ID'] = data['Serial']
data['Brand'] = data['Brand_ID'].map(meta.variable_value_labels['Brand_ID'])
data['Round'] = "R01" # data['Q_Ola'].map(meta.variable_value_labels['Q_Ola'])
data['Sex'] = data['SEX_SPSS'].map(meta.variable_value_labels['SEX_SPSS'])
data['Sec'] = data['NSE_SPSS'].map(meta.variable_value_labels['NSE_SPSS'])
data['Age'] = data['EDAD_SPSS']
data['Region'] = data['Location_SPSS'].map(meta.variable_value_labels['Location_SPSS'])

# Preprocesamiento 
data['MDF_DemandPower'] = data['Power_SPSS']
data['MDF_Meaningful'] = data['M_F_I_SPSS']
data['MDF_Different'] = data['D_F_I_SPSS']
data['MDF_Salient'] = data['S_F_I_SPSS']
data['MDIN_TotalUnaidedAwareness'] = data['UN_AWARE_SPSS']
data['BE2A_T2B_Frequent'] = (data['FAM_FMCG_SPSS'] <= 2).astype(int)
data['BE2A_T4B_Trial'] = (data['FAM_FMCG_SPSS'] <= 4).astype(int)
data['BE2A_T5B_Aware'] = (data['FAM_FMCG_SPSS'] <= 5).astype(int)
data['BE3_T2B_Consideration'] = (data['CONS_SPSS'] <= 2).astype(int)
data['BE6_T2B_Affinity'] = (data['AFFINITY_DA_SPSS'] >= 6).astype(int) # & (data['BE6_Aware'] < 8)
data['BP1_T2B_Unique'] = (data['UNIQUE_DA_SPSS'] >= 6).astype(int) # & (data['BP1_Aware'] < 8)
data['BP2_T2B_MeetNeeds'] = (data['MEETS_NEEDS_DA_SPSS'] >= 6).astype(int) # & (data['BP2_Aware'] < 8)
data['BP3_T2B_Dynamic'] = (data['DYNAMIC_DA_SPSS'] >= 6).astype(int) # & (data['BP3_Aware'] < 8)
data['BP5_T2B_Price'] = (data['PRICE_DA_SPSS'] >= 6).astype(int) # & (data['BP5_Aware'] < 8)
data['BP6_TB_Worth'] = (data['WORTH_SPSS'] == 3).astype(int)

# Si no hay ponderador, añadir un ponderador dummy 
data['Weight'] = 1

# Aplicar etiquetas a variables 
labels= {
    'BD7_SPSS__1': 'OCC_Eating_a_meal_or_a_snack_at_home',
    'BD7_SPSS__2': 'OCC_Eating_a_meal_or_a_snack_out_of_home',
    'BD7_SPSS__3': 'OCC_Doing_everyday_activities',
    'BD7_SPSS__4': 'OCC_Relaxing_unwinding_alone',
    'BD7_SPSS__5': 'OCC_Relaxing_unwinding_with_others',
    'BD7_SPSS__6': 'OCC_Having_an_evening_or_pre_dinner_drink_before_your_meal',
    'BD7_SPSS__7': 'OCC_Socializing_with_male_and_female_friendsfamily_at_home',
    'BD7_SPSS__8': 'OCC_Socializing_with_male_and_female_friendsfamily_out_of_home',
    'BD7_SPSS__9': 'OCC_Socializing_with_male_friendsfamily_at_home',
    'BD7_SPSS__10': 'OCC_Socializing_with_male_friendsfamily_out_of_home',
    'BD7_SPSS__11': 'OCC_Attending_a_party_or_celebration_at_home',
    'BD7_SPSS__12': 'OCC_Attending_a_party_or_celebration_out_of_home',
    'TBCA_Eventos_SPSS_1_slice': 'AWE_Corona_Capital_CDMX',
    'TBCA_Eventos_SPSS_2_slice': 'AWE_Corona_Capital_Guadalajara',
    'TBCA_Eventos_SPSS_3_slice': 'AWE_Corona_Capital_Monterrey',
    'TBCA_Eventos_SPSS_4_slice': 'AWE_Corona_Capital_Merida',
    'TBCA_Eventos_SPSS_5_slice': 'AWE_Electric_Daisy_Carnival_EDC',
    'TBCA_Eventos_SPSS_6_slice': 'AWE_Tecate_Emblema_CDMX',
    'TBCA_Eventos_SPSS_7_slice': 'AWE_Tecate_Pal_Norte',
    'TBCA_Eventos_SPSS_8_slice': 'AWE_Vive_Latino',
    'TBCA_Eventos_SPSS_9_slice': 'AWE_Feria_de_San_Marcos',
    'TBCA_Eventos_SPSS_10_slice': 'AWE_NBA_México',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_1_slice': 'TBCAE_Corona_Capital_CDMX_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_2_slice': 'TBCAE_Corona_Capital_CDMX_In_a_store',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_3_slice': 'TBCAE_Corona_Capital_CDMX_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_4_slice': 'TBCAE_Corona_Capital_CDMX_TV',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_5_slice': 'TBCAE_Corona_Capital_CDMX_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_6_slice': 'TBCAE_Corona_Capital_CDMX_Radio',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_7_slice': 'TBCAE_Corona_Capital_CDMX_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_8_slice': 'TBCAE_Corona_Capital_CDMX_Social_Media',
    'TBCA_Eventos_Medios_SPSS_1_TBCA_Eventos_Medios_9_slice': 'TBCAE_Corona_Capital_CDMX_Online_Video',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_1_slice': 'TBCAE_Corona_Capital_Guadalajara_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_2_slice': 'TBCAE_Corona_Capital_Guadalajara_In_a_store',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_3_slice': 'TBCAE_Corona_Capital_Guadalajara_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_4_slice': 'TBCAE_Corona_Capital_Guadalajara_TV',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_5_slice': 'TBCAE_Corona_Capital_Guadalajara_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_6_slice': 'TBCAE_Corona_Capital_Guadalajara_Radio',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_7_slice': 'TBCAE_Corona_Capital_Guadalajara_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_8_slice': 'TBCAE_Corona_Capital_Guadalajara_Social_Media',
    'TBCA_Eventos_Medios_SPSS_2_TBCA_Eventos_Medios_9_slice': 'TBCAE_Corona_Capital_Guadalajara_Online_Video',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_1_slice': 'TBCAE_Corona_Capital_Monterrey_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_2_slice': 'TBCAE_Corona_Capital_Monterrey_In_a_store',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_3_slice': 'TBCAE_Corona_Capital_Monterrey_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_4_slice': 'TBCAE_Corona_Capital_Monterrey_TV',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_5_slice': 'TBCAE_Corona_Capital_Monterrey_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_6_slice': 'TBCAE_Corona_Capital_Monterrey_Radio',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_7_slice': 'TBCAE_Corona_Capital_Monterrey_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_8_slice': 'TBCAE_Corona_Capital_Monterrey_Social_Media',
    'TBCA_Eventos_Medios_SPSS_3_TBCA_Eventos_Medios_9_slice': 'TBCAE_Corona_Capital_Monterrey_Online_Video',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_1_slice': 'TBCAE_Corona_Capital_Merida_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_2_slice': 'TBCAE_Corona_Capital_Merida_In_a_store',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_3_slice': 'TBCAE_Corona_Capital_Merida_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_4_slice': 'TBCAE_Corona_Capital_Merida_TV',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_5_slice': 'TBCAE_Corona_Capital_Merida_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_6_slice': 'TBCAE_Corona_Capital_Merida_Radio',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_7_slice': 'TBCAE_Corona_Capital_Merida_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_8_slice': 'TBCAE_Corona_Capital_Merida_Social_Media',
    'TBCA_Eventos_Medios_SPSS_4_TBCA_Eventos_Medios_9_slice': 'TBCAE_Corona_Capital_Merida_Online_Video',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_1_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_2_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_In_a_store',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_3_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_4_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_TV',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_5_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_6_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_Radio',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_7_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_8_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_Social_Media',
    'TBCA_Eventos_Medios_SPSS_5_TBCA_Eventos_Medios_9_slice': 'TBCAE_Electric_Daisy_Carnival_EDC_Online_Video',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_1_slice': 'TBCAE_Tecate_Emblema_CDMX_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_2_slice': 'TBCAE_Tecate_Emblema_CDMX_In_a_store',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_3_slice': 'TBCAE_Tecate_Emblema_CDMX_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_4_slice': 'TBCAE_Tecate_Emblema_CDMX_TV',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_5_slice': 'TBCAE_Tecate_Emblema_CDMX_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_6_slice': 'TBCAE_Tecate_Emblema_CDMX_Radio',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_7_slice': 'TBCAE_Tecate_Emblema_CDMX_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_8_slice': 'TBCAE_Tecate_Emblema_CDMX_Social_Media',
    'TBCA_Eventos_Medios_SPSS_6_TBCA_Eventos_Medios_9_slice': 'TBCAE_Tecate_Emblema_CDMX_Online_Video',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_1_slice': 'TBCAE_Tecate_Pal_Norte_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_2_slice': 'TBCAE_Tecate_Pal_Norte_In_a_store',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_3_slice': 'TBCAE_Tecate_Pal_Norte_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_4_slice': 'TBCAE_Tecate_Pal_Norte_TV',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_5_slice': 'TBCAE_Tecate_Pal_Norte_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_6_slice': 'TBCAE_Tecate_Pal_Norte_Radio',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_7_slice': 'TBCAE_Tecate_Pal_Norte_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_8_slice': 'TBCAE_Tecate_Pal_Norte_Social_Media',
    'TBCA_Eventos_Medios_SPSS_7_TBCA_Eventos_Medios_9_slice': 'TBCAE_Tecate_Pal_Norte_Online_Video',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_1_slice': 'TBCAE_Vive_Latino_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_2_slice': 'TBCAE_Vive_Latino_In_a_store',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_3_slice': 'TBCAE_Vive_Latino_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_4_slice': 'TBCAE_Vive_Latino_TV',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_5_slice': 'TBCAE_Vive_Latino_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_6_slice': 'TBCAE_Vive_Latino_Radio',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_7_slice': 'TBCAE_Vive_Latino_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_8_slice': 'TBCAE_Vive_Latino_Social_Media',
    'TBCA_Eventos_Medios_SPSS_8_TBCA_Eventos_Medios_9_slice': 'TBCAE_Vive_Latino_Online_Video',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_1_slice': 'TBCAE_Feria_de_San_Marcos_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_2_slice': 'TBCAE_Feria_de_San_Marcos_In_a_store',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_3_slice': 'TBCAE_Feria_de_San_Marcos_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_4_slice': 'TBCAE_Feria_de_San_Marcos_TV',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_5_slice': 'TBCAE_Feria_de_San_Marcos_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_6_slice': 'TBCAE_Feria_de_San_Marcos_Radio',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_7_slice': 'TBCAE_Feria_de_San_Marcos_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_8_slice': 'TBCAE_Feria_de_San_Marcos_Social_Media',
    'TBCA_Eventos_Medios_SPSS_9_TBCA_Eventos_Medios_9_slice': 'TBCAE_Feria_de_San_Marcos_Online_Video',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_1_slice': 'TBCAE_NBA_México_BillboardOutdoor_Advertising',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_2_slice': 'TBCAE_NBA_México_In_a_store',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_3_slice': 'TBCAE_NBA_México_In_a_barRestaurant',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_4_slice': 'TBCAE_NBA_México_TV',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_5_slice': 'TBCAE_NBA_México_Paid_TV',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_6_slice': 'TBCAE_NBA_México_Radio',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_7_slice': 'TBCAE_NBA_México_NewspapersMagazines',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_8_slice': 'TBCAE_NBA_México_Social_Media',
    'TBCA_Eventos_Medios_SPSS_10_TBCA_Eventos_Medios_9_slice': 'TBCAE_NBA_México_Online_Video',
    'ASIS_EVENTOS_SPSS__1': 'ASISE_Corona_Capital_CDMX',
    'ASIS_EVENTOS_SPSS__2': 'ASISE_Corona_Capital_Guadalajara',
    'ASIS_EVENTOS_SPSS__3': 'ASISE_Corona_Capital_Monterrey',
    'ASIS_EVENTOS_SPSS__4': 'ASISE_Corona_Capital_Merida',
    'ASIS_EVENTOS_SPSS__5': 'ASISE_Electric_Daisy_Carnival_EDC',
    'ASIS_EVENTOS_SPSS__6': 'ASISE_Tecate_Emblema_CDMX',
    'ASIS_EVENTOS_SPSS__7': 'ASISE_Tecate_Pal_Norte',
    'ASIS_EVENTOS_SPSS__8': 'ASISE_Vive_Latino',
    'ASIS_EVENTOS_SPSS__9': 'ASISE_Feria_de_San_Marcos',
    'ASIS_EVENTOS_SPSS__10': 'ASISE_NBA_México',
    'PATROCINADORES_SPSS_1_slice': 'SPONE_Corona_Capital_CDMX',
    'PATROCINADORES_SPSS_2_slice': 'SPONE_Corona_Capital_Guadalajara',
    'PATROCINADORES_SPSS_3_slice': 'SPONE_Corona_Capital_Monterrey',
    'PATROCINADORES_SPSS_4_slice': 'SPONE_Corona_Capital_Merida',
    'PATROCINADORES_SPSS_5_slice': 'SPONE_Electric_Daisy_Carnival_EDC',
    'PATROCINADORES_SPSS_6_slice': 'SPONE_Tecate_Emblema_CDMX',
    'PATROCINADORES_SPSS_7_slice': 'SPONE_Tecate_Pal_Norte',
    'PATROCINADORES_SPSS_8_slice': 'SPONE_Vive_Latino',
    'PATROCINADORES_SPSS_9_slice': 'SPONE_Feria_de_San_Marcos',
    'PATROCINADORES_SPSS_10_slice': 'SPONE_NBA_México',
    'TOUCHPOINT_REACH_SPSS__1': 'TP_Billboard_Outdoor_Advertising',
    'TOUCHPOINT_REACH_SPSS__2': 'TP_In_a_store',
    'TOUCHPOINT_REACH_SPSS__3': 'TP_In_a_bar_restaurant',
    'TOUCHPOINT_REACH_SPSS__4': 'TP_TV',
    'TOUCHPOINT_REACH_SPSS__5': 'TP_Paid_TV_Non_Sports_Programming',
    'TOUCHPOINT_REACH_SPSS__6': 'TP_Radio',
    'TOUCHPOINT_REACH_SPSS__7': 'TP_Newspapers_magazines',
    'TOUCHPOINT_REACH_SPSS__8': 'TP_Social_Media',
    'TOUCHPOINT_REACH_SPSS__9': 'TP_Online_Video',
    'IMAGERY_SPSS__1': 'IMG_Has_a_smooth_taste',
    'IMAGERY_SPSS__2': 'IMG_Is_a_high_quality_brand',
    'IMAGERY_SPSS__3': 'IMG_Is_good_to_be_seen_drinking',
    'IMAGERY_SPSS__4': 'IMG_Is_good_to_drink_when_you_want_to_relax',
    'IMAGERY_SPSS__5': 'IMG_Is_more_refreshing_than_others',
    'IMAGERY_SPSS__6': 'IMG_Makes_an_occasion_more_special',
    'IMAGERY_SPSS__7': 'IMG_You_like_to_savour',
    'IMAGERY_SPSS__8': 'IMG_You_like_to_share_with_your_friends',
    'IMAGERY_SPSS__9': 'IMG_Is_sophisticated',
    'IMAGERY_SPSS__10': 'IMG_Adds_energy_to_your_social_occasion',
    'IMAGERY_SPSS__11': 'IMG_Goes_well_with_food_meals',
    'IMAGERY_SPSS__12': 'IMG_Represents_a_balanced_lifestyle_choice',
    'IMAGERY_SPSS__13': 'IMG_Is_a_successful_international_brand',
    'IMAGERY_SPSS__14': 'IMG_Is_brewed_with_care',
    'IMAGERY_SPSS__15': 'IMG_Represents_my_national_pride',
    'IMAGERY_SPSS__16': 'IMG_Is_the_beer_that_combines_the_best_with_mexican_flavors',
    'IMAGERY_SPSS__17': 'IMG_Is_made_from_natural_ingredients',
    'IMAGERY_SPSS__18': 'IMG_It_is_a_socially_responsible_brand',
    'IMAGERY_SPSS__19': 'IMG_It_is_open_and_honest_about_what_it_is_made_of',
    'IMAGERY_SPSS__20': 'IMG_Is_a_brand_that_represents_our_traditions_and_heritage',
    'IMAGERY_SPSS__21': 'IMG_A_brand_for_those_who_enjoy_the_outdoors',
    'ACT1_SPSS__1': 'ACT_It_was_at_a_good_price',
    'ACT1_SPSS__2': 'ACT_It_was_on_sale_special_or_promotion',
    'ACT1_SPSS__3': 'ACT_It_had_promotional_material_displayed',
    'ACT1_SPSS__4': 'ACT_It_was_in_a_different_section_of_the_store_than_usual',
    'ACT1_SPSS__5': 'ACT_It_was_easy_to_find',
    'ACT1_SPSS__6': 'ACT_It_included_a_gift_or_free_sample',
    'ACT1_SPSS__7': 'ACT_It_was_refrigerated',
    'ACT1_SPSS__8': 'ACT_It_was_a_new_product',
    'ACT1_SPSS__9': 'ACT_It_was_recommended_to_me',
    'ACT1_SPSS__10': 'ACT_It_came_in_the_pack_size_I_wanted_and_others_didnt',
    'ACT1_SPSS__11': 'ACT_Other_brands_I_usually_buy_were_not_available',
    'ACT1_SPSS__12': 'ACT_It_was_featured_or_listed_on_a_menu',
    'ACT1_SPSS__13': 'ACT_It_was_easily_available_at_the_places_I_go_to',
    'ACT1_SPSS__14': 'ACT_It_came_in_the_presentation_that_I_was_looking_for',
    'ACT1_SPSS__15': 'ACT_Sponsors_the_team_or_sports_I_like',
    'EXPERIENCE_MEDIO_1_SPSS': 'TPEXP_At_a_music_venue_music_event',
    'EXPERIENCE_MEDIO_2_SPSS': 'TPEXP_In_a_store',
    'EXPERIENCE_MEDIO_3_SPSS': 'TPEXP_In_a_bar_restaurant',
    'EXPERIENCE_MEDIO_4_SPSS': 'TPEXP_TV',
    'EXPERIENCE_MEDIO_5_SPSS': 'TPEXP_Newspapers_magazines',
    'EXPERIENCE_MEDIO_6_SPSS': 'TPEXP_Billboard_Outdoor_Advertising',
    'EXPERIENCE_MEDIO_7_SPSS': 'TPEXP_Social_Media',
    'EXPERIENCE_MEDIO_8_SPSS': 'TPEXP_Online_Video',
    'EXPERIENCE_MEDIO_9_SPSS': 'TPEXP_At_a_sporting_event_stadium',
    'EXPERIENCE_MEDIO_10_SPSS': 'TPEXP_Paid_TV',
    'TBCA_SPSS__1': 'TBCA_At_a_music_venue_music_event',
    'TBCA_SPSS__2': 'TBCA_In_a_store',
    'TBCA_SPSS__3': 'TBCA_In_a_bar_restaurant',
    'TBCA_SPSS__4': 'TBCA_TV',
    'TBCA_SPSS__5': 'TBCA_Newspapers_magazines',
    'TBCA_SPSS__6': 'TBCA_Billboard_Outdoor_Advertising',
    'TBCA_SPSS__7': 'TBCA_Social_Media',
    'TBCA_SPSS__8': 'TBCA_Online_Video',
    'TBCA_SPSS__9': 'TBCA_At_a_sporting_event_stadium',
    'TBCA_SPSS__10': 'TBCA_Paid_TV',
    'SP1_1.0': 'SP1_Online',
    'SP1_2.0': 'SP1_Depositos_de_bebidas',
    'SP1_3.0': 'SP1_Supermercado',
    'SP1_4.0': 'SP1_Modelorama_o_Tecate_Six',
    'SP1_5.0': 'SP1_Antro',
    'SP1_6.0': 'SP1_Restaurante',
    'SP1_7.0': 'SP1_Bar',
    'SP1_8.0': 'SP1_Tienda_de_la_esquina',
    'SP1_9.0': 'SP1_Conveniencia',
    'SP1_10.0': 'SP1_Otro_tipo_de_tienda_no_listado',
    'SP1_11.0': 'SP1_No_recuerdo',
}

data.columns = data.columns.map(lambda col: labels.get(col, col))

# Filtros principales 
data_aware = data[data['FAM_FMCG_SPSS'] <= 5].copy()
data_filtered = data[(data['Round'] == 'R01') & (data['FILTRO_SPSS'] != 1)].copy() # & (data['AWE_Corona_Capital_CDMX']==1) 
data_aware_filtered = data_aware[(data_aware['Round'] == 'R01') & (data['FILTRO_SPSS'] != 1)].copy() # & (data['AWE_Corona_Capital_CDMX']==1) 
data_brand_aware_filtered = data_aware_filtered[
    (data_aware_filtered['Brand'] == 'Corona Golden Light') | 
    (data_aware_filtered['Brand'] == 'Corona Extra') |
    (data_aware_filtered['Brand'] == 'Corona Cero') |
    (data_aware_filtered['Brand'] == 'Corona Light')
    ].fillna(0).copy()

C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26260\504618206.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Respondent_ID'] = data['Serial']
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26260\504618206.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Brand'] = data['Brand_ID'].map(meta.variable_value_labels['Brand_ID'])
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26260\504618206.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

In [5]:
# Calculos del preanalisis 
out_describe_dataset = data.describe(include='all').T
out_valid_counts = data.groupby('Brand').count().T
out_valid_counts_percentage = out_valid_counts / out_valid_counts.max()
out_aware_base_by_round = pd.crosstab(data_aware['Round'], data_aware['Brand']) 
out_aware_base_by_round_perc = pd.crosstab(data_aware['Round'], data_aware['Brand'])  / pd.crosstab(data['Round'], data['Brand']) 
out_total_base_by_round = pd.crosstab(data['Round'], data['Brand'])
out_results_total_base = data_filtered.groupby('Brand').mean(numeric_only=True).T
out_results_total_base_counts = data_filtered.groupby('Brand').sum(numeric_only=True).T
out_valid_total_base = data_filtered.groupby('Brand').count().apply(lambda row: row / row.max(), axis=1).T
out_results_aware_base = data_aware_filtered.groupby('Brand').mean(numeric_only=True).T
out_results_aware_base_counts = data_aware_filtered.groupby('Brand').sum(numeric_only=True).T
out_valid_aware_base = data_aware_filtered.groupby('Brand').count().apply(lambda row: row / row.max(), axis=1).T
out_correlations_brand_aware = data_brand_aware_filtered.corr(numeric_only=True)
out_results_aware_weighed = brand_weighted_mean(data_brand_aware_filtered, 'Weight').to_frame()

demos_list = ['Sex', 'Age', 'Region', 'Sec']
out_demos_counts = pd.DataFrame()
for demo in demos_list:
    temp = (data[demo].value_counts() / data[demo].count()).to_frame()
    temp.index.name = 'Demo'
    out_demos_counts = pd.concat([out_demos_counts, temp])

In [6]:
with pd.ExcelWriter('Preanalisis Output.xlsx') as writer:
    out_demos_counts.to_excel(writer, sheet_name='Demographics')
    out_describe_dataset.to_excel(writer, sheet_name='Describe')
    out_valid_counts.to_excel(writer, sheet_name='Valid Counts')
    out_valid_counts_percentage.to_excel(writer, sheet_name='Valid Counts_Perc')
    out_aware_base_by_round.to_excel(writer, sheet_name='Brand Awareness Wave')
    out_results_total_base.to_excel(writer, sheet_name='Results Total_Mean')
    out_results_total_base_counts.to_excel(writer, sheet_name='Results Total_Counts')
    out_results_aware_base.to_excel(writer, sheet_name='Results Aware_Mean')
    out_results_aware_base_counts.to_excel(writer, sheet_name='Results Aware_Counts')
    out_results_aware_weighed.to_excel(writer, sheet_name='Results Aware_Weighted Mean')
    out_correlations_brand_aware.to_excel(writer, sheet_name='Correlations brand aware base')

In [7]:
# Selección de variables para DF final 
vars_filter = [
    'Respondent_ID',
    'Brand',
    'Round',
    'Sex',
    'Sec',    
    'Age',
    'Region',
    'MDF_DemandPower',
    'MDF_Meaningful',
    'MDF_Different',
    'MDF_Salient',
    'MDIN_TotalUnaidedAwareness',
    'BE2A_T2B_Frequent',
    'BE2A_T4B_Trial',
    'BE2A_T5B_Aware',
    'BE3_T2B_Consideration',
    'BE6_T2B_Affinity',
    'BP1_T2B_Unique',
    'BP2_T2B_MeetNeeds',
    'BP3_T2B_Dynamic',
    'BP5_T2B_Price',
    'BP6_TB_Worth',
    'Weight'
    ]

vars_filter.extend(data.filter(like='IMG_').columns)
vars_filter.extend(data.filter(like='AWE_').columns)
vars_filter.extend(data.filter(like='TBCAE_').columns)
vars_filter.extend(data.filter(like='ASISE_').columns)
vars_filter.extend(data.filter(like='SPONE_').columns)
vars_filter.extend(data.filter(like='TP_').columns)
vars_filter.extend(data.filter(like='ACT_').columns)
vars_filter.extend(data.filter(like='TPEXP_').columns)
vars_filter.extend(data.filter(like='TBCA_').columns)
vars_filter.extend(data.filter(like='OCC_').columns)
vars_filter.extend(data.filter(like='SP1_').columns)

# DF Filtrado 
data_output = data_brand_aware_filtered[vars_filter].copy()

In [8]:
# Salida en SPSS
pyreadstat.write_sav(df=data_brand_aware_filtered[vars_filter], dst_path='Brand_Aware_Filtered_DF.sav', variable_value_labels=meta.variable_value_labels)

# Salida en CSV
data_output.to_csv('Brand_Aware_Filtered_DF.csv', index=False)